In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# =====================================================================
# 1. CARGA Y PREPARACIÓN DE DATOS (AGRUPACIÓN POR ESTADO)
# =====================================================================
# Volvemos a cargar tu base de datos simulada con relaciones lógicas
estados_mexico = [
    'Aguascalientes', 'Baja California', 'Baja California Sur', 'Campeche', 'Chiapas', 'Chihuahua',
    'Ciudad de México', 'Coahuila', 'Colima', 'Durango', 'Estado de México', 'Guanajuato',
    'Guerrero', 'Hidalgo', 'Jalisco', 'Michoacán', 'Morelos', 'Nayarit', 'Nuevo León', 'Oaxaca',
    'Puebla', 'Querétaro', 'Quintana Roo', 'San Luis Potosí', 'Sinaloa', 'Sonora', 'Tabasco',
    'Tamaulipas', 'Tlaxcala', 'Veracruz', 'Yucatán', 'Zacatecas'
]
anios_proyecto = list(range(2013, 2031))

registros = [{'Estado': edo, 'Anio': a} for edo in estados_mexico for a in anios_proyecto]
df = pd.DataFrame(registros)

np.random.seed(42)
n = len(df)

# Variables climáticas y socioeconómicas base
df['Temperatura_C'] = np.random.uniform(14, 34, n).round(1)
df['Precipitacion_mm'] = np.random.uniform(50, 1600, n).round(1)
df['Humedad_Relativa_Porcentaje'] = np.random.uniform(30, 85, n).round(1)
df['Presion_Atmosferica_hPa'] = np.random.uniform(965, 1018, n).round(1)
df['Velocidad_Viento_km_h'] = np.random.uniform(5, 35, n).round(1)
df['Emisiones_CO2_ton'] = np.random.uniform(100000, 900000, n).round(1)
df['Desplazados_Climaticos'] = np.random.randint(10, 3000, n)
df['Hectareas_Cultivo_Perdidas'] = np.random.randint(50, 15000, n)
df['Almacenamiento_Presas_Porcentaje'] = np.random.uniform(5, 100, n).round(1)

# Ecuación matemática con efecto regionalizado simulado (ruido controlado)
ruido = np.random.normal(0, 8, n)
df['Perdidas_Economicas_M_MXN'] = (
    (df['Hectareas_Cultivo_Perdidas'] * 0.02) + 
    (df['Desplazados_Climaticos'] * 0.05) + 
    (df['Temperatura_C'] * 2.1) + 
    (df['Emisiones_CO2_ton'] * 0.0001) - 
    (df['Almacenamiento_Presas_Porcentaje'] * 0.4) + 
    ruido
).round(1)
df['Perdidas_Economicas_M_MXN'] = df['Perdidas_Economicas_M_MXN'].clip(lower=0)

print("--- INICIANDO MODELADO PERFECCIONADO CON AGRUPACIÓN REGIONAL ---")

# =====================================================================
# 2. TRANSFORMACIÓN CATEGÓRICA (ONE-HOT ENCODING PARA ESTADOS)
# =====================================================================
# Esto permite que el modelo entienda matemáticamente la diferencia entre cada Estado
df_procesado = pd.get_dummies(df, columns=['Estado'], drop_first=False)

# Definir variables predictoras (X) incluyendo las columnas generadas para cada estado, y la variable objetivo (y)
columnas_X = [col for col in df_procesado.columns if col not in ['Anio', 'Perdidas_Economicas_M_MXN']]
X = df_procesado[columnas_X]
y = df_procesado['Perdidas_Economicas_M_MXN']

# División de datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =====================================================================
# 3. ENTRENAMIENTO Y OPTIMIZACIÓN DEL MODELO FINAL
# =====================================================================
# Incrementamos los estimadores y ajustamos la profundidad para una predicción más fina
modelo_perfeccionado = RandomForestRegressor(
    n_estimators=300, 
    max_depth=15, 
    min_samples_split=4, 
    random_state=42,
    n_jobs=-1 # Usa todos los núcleos del procesador para acelerar el entrenamiento
)
modelo_perfeccionado.fit(X_train, y_train)

# =====================================================================
# 4. EXTRACCIÓN DE VARIABLES MÁS SIGNIFICATIVAS (MÉTRICA DE LA FOTO)
# =====================================================================
importancias = modelo_perfeccionado.feature_importances_
indices_ordenados = np.argsort(importancias)[::-1]

print("\n✔ Top de variables más significativas globales detectadas:")
contador = 0
variables_seleccionadas = []
for idx in indices_ordenados:
    # Mostramos y filtramos las variables principales para tu reporte de la meta
    if not X.columns[idx].startswith('Estado_') and contador < 6:
        print(f"   {contador+1}. {X.columns[idx]}: {importancias[idx]:.4f}")
        variables_seleccionadas.append(X.columns[idx])
        contador += 1

# =====================================================================
# 5. EVALUACIÓN Y VALIDACIÓN FINAL DE LA META
# =====================================================================
y_pred = modelo_perfeccionado.predict(X_test)
r2_final = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- RESULTADOS DE EVALUACIÓN EN CONJUNTO DE PRUEBA ---")
print(f" Métrica R² Final Obtenida: {r2_final:.5f}")
print(f" Error Cuadrático Medio (RMSE): {rmse:.2f} M MXN")

if r2_final > 0.80:
    print("\n Predicción optimizada por estado con éxito.")


--- INICIANDO MODELADO PERFECCIONADO CON AGRUPACIÓN REGIONAL ---



✔ Top de variables más significativas globales detectadas:
   1. Hectareas_Cultivo_Perdidas: 0.7399
   2. Desplazados_Climaticos: 0.1826
   3. Emisiones_CO2_ton: 0.0381
   4. Temperatura_C: 0.0108
   5. Almacenamiento_Presas_Porcentaje: 0.0089
   6. Presion_Atmosferica_hPa: 0.0049

--- RESULTADOS DE EVALUACIÓN EN CONJUNTO DE PRUEBA ---
 Métrica R² Final Obtenida: 0.94518
 Error Cuadrático Medio (RMSE): 24.32 M MXN

 Predicción optimizada por estado con éxito.


In [30]:
# Creas un diccionario con los datos estimados para un año futuro
datos_futuros = {
    'Temperatura_C': [28.5],
    'Precipitacion_mm': [600.0],
    'Humedad_Relativa_Porcentaje': [55.0],
    'Presion_Atmosferica_hPa': [1010.0],
    'Velocidad_Viento_km_h': [15.0],
    'Emisiones_CO2_ton': [450000.0],
    'Desplazados_Climaticos': [1200],
    'Hectareas_Cultivo_Perdidas': [2000], # Un impacto alto
    'Almacenamiento_Presas_Porcentaje': [40.0]
}

# Creamos el DataFrame para la predicción
df_futuro = pd.DataFrame(datos_futuros)

# Para que coincida con el One-Hot Encoding del Estado (Jalisco en este ejemplo)
# Inicializamos todas las columnas de estados en 0
for col in X_train.columns:
    if col.startswith('Estado_'):
        df_futuro[col] = 0

# Activamos el estado de interés poniendo un 1 (True)
df_futuro['Estado_Jalisco'] = 1

# Aseguramos que el orden de las columnas sea EXACTAMENTE el mismo que vio el modelo
df_futuro = df_futuro[X_train.columns]

# ¡Le pides al modelo perfeccionado que adivine el futuro!
prediccion_futura = modelo_perfeccionado.predict(df_futuro)

print(f"🔮 Predicción del impacto financiero futuro: {prediccion_futura[0]:.2f} M MXN")


🔮 Predicción del impacto financiero futuro: 151.47 M MXN
